In [ ]:
# %pip install pyspark

import datetime as dt
from functools import reduce
from pathlib import Path
import zipfile

from pyspark import sql
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = sql.SparkSession.builder.appName("dbscoring-spark-lab").getOrCreate()

colab_sources_dir = Path("/content/dbscoring/data/sources")
if not colab_sources_dir.exists():
    for zip_path in (Path("/content/data.zip"), Path("data.zip")):
        if zip_path.exists():
            extract_root = Path("/content/dbscoring")
            extract_root.mkdir(parents=True, exist_ok=True)
            if zipfile.is_zipfile(zip_path):
                with zipfile.ZipFile(file=zip_path) as archive:
                    archive.extractall(extract_root)
                print(f"Extracted {zip_path} to {extract_root}")
            else:
                print(f"Skip unpack: {zip_path} is not a valid zip archive")
            break
else:
    print(f"Skip unpack: {colab_sources_dir} already exists")

for candidate in (Path("/content/dbscoring/data/sources"), Path("data/sources"), Path("../data/sources")):
    if candidate.exists():
        BASE_DIR = candidate
        break
else:
    BASE_DIR = Path("data/sources")

for candidate in (Path("/content/dbscoring/data/warehouse_spark"), Path("data/warehouse_spark"), Path("../data/warehouse_spark")):
    if candidate.exists() or candidate.parent.exists():
        WAREHOUSE_ROOT = candidate
        break
else:
    WAREHOUSE_ROOT = Path("data/warehouse_spark")

NOW_TS = dt.datetime.now().replace(microsecond=0)
START_TS = dt.datetime(2023, 1, 1, 0, 0, 0)
FUTURE_TS = dt.datetime(9999, 12, 31, 0, 0, 0)
SOURCE_REGISTRY = {
    "client_cards_daily": {
        "source_id": 1,
        "source_description": "Daily client card source.",
        "update_frequency": "daily",
        "target_table": "client_daily_attrs_scd2",
        "columns": (
            "srv_mb_nflag",
            "cc_stoplist",
            "lne_tot_debt_int_ovrd_rub_amt",
            "lne_tot_debt_ovrd_rub_amt",
        ),
    },
    "credit_cards_info": {
        "source_id": 2,
        "source_description": "Monthly credit card source.",
        "update_frequency": "monthly",
        "target_table": "client_monthly_attrs_scd1",
        "columns": (
            "client_income_amt",
            "oi_total_amt",
            "act_pl_os_rub_amt",
            "payroll_client_nflag",
            "inf_payroll_rub_amt",
            "legal_entity_amt",
            "inc_avg_risk_rub_amt",
            "otf_loan_rub_amt",
            "otf_fee_rub_amt",
            "inf_transfer_rub_amt",
            "cc_ever_nflag",
        ),
    },
    "deb_cards_info": {
        "source_id": 3,
        "source_description": "Monthly debit card source.",
        "update_frequency": "monthly",
        "target_table": "client_monthly_attrs_scd1",
        "columns": (
            "onl_bank_active_1m_nfalg",
            "auto_pay_active_qty",
            "cl_income_1m_amt",
            "dep_acc_1st_open_dt",
            "wdr_cash_6m_amt",
            "cash_op_6m_amt",
            "cash_3m_qty",
            "lst_balance_amt",
            "card_active_1m_nflag",
        ),
    },
}
ALL_COLUMNS = tuple(col for meta in SOURCE_REGISTRY.values() for col in meta["columns"])
ATTR_ID_BY_NAME = {name: idx for idx, name in enumerate(ALL_COLUMNS, start=1)}


In [ ]:
# Physical schemas from the ER diagram.
dim_sources_schema = T.StructType([
    T.StructField(name="source_id", dataType=T.IntegerType(), nullable=False),
    T.StructField(name="source_name", dataType=T.StringType(), nullable=False),
    T.StructField(name="source_description", dataType=T.StringType(), nullable=True),
    T.StructField(name="update_frequency", dataType=T.StringType(), nullable=True),
    T.StructField(name="row_create_dtime", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="row_update_dtime", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="valid_from", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="valid_to", dataType=T.TimestampType(), nullable=True),
])

dim_attributes_schema = T.StructType([
    T.StructField(name="attribute_id", dataType=T.IntegerType(), nullable=False),
    T.StructField(name="attribute_name", dataType=T.StringType(), nullable=False),
    T.StructField(name="attribute_description", dataType=T.StringType(), nullable=True),
    T.StructField(name="data_type", dataType=T.StringType(), nullable=True),
    T.StructField(name="source_id", dataType=T.IntegerType(), nullable=True),
    T.StructField(name="update_frequency", dataType=T.StringType(), nullable=True),
    T.StructField(name="row_create_dtime", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="row_update_dtime", dataType=T.TimestampType(), nullable=True),
])

load_log_schema = T.StructType([
    T.StructField(name="load_id", dataType=T.LongType(), nullable=False),
    T.StructField(name="source_id", dataType=T.IntegerType(), nullable=True),
    T.StructField(name="source_report_dt", dataType=T.StringType(), nullable=True),
    T.StructField(name="load_start_dtime", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="load_end_dtime", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="target_table", dataType=T.StringType(), nullable=True),
    T.StructField(name="load_status", dataType=T.StringType(), nullable=True),
    T.StructField(name="rows_loaded", dataType=T.LongType(), nullable=True),
    T.StructField(name="error_message", dataType=T.StringType(), nullable=True),
])

monthly_schema = T.StructType([
    T.StructField(name="client_id", dataType=T.IntegerType(), nullable=False),
    T.StructField(name="attribute_id", dataType=T.IntegerType(), nullable=False),
    T.StructField(name="report_dt", dataType=T.StringType(), nullable=False),
    T.StructField(name="attribute_value", dataType=T.StringType(), nullable=True),
    T.StructField(name="source_id", dataType=T.IntegerType(), nullable=True),
    T.StructField(name="row_update_dtime", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="row_loading_id", dataType=T.LongType(), nullable=True),
    T.StructField(name="row_hash_val", dataType=T.StringType(), nullable=True),
])

daily_schema = T.StructType([
    T.StructField(name="client_id", dataType=T.IntegerType(), nullable=False),
    T.StructField(name="attribute_id", dataType=T.IntegerType(), nullable=False),
    T.StructField(name="attribute_value", dataType=T.StringType(), nullable=True),
    T.StructField(name="row_actual_from", dataType=T.StringType(), nullable=False),
    T.StructField(name="row_actual_to", dataType=T.StringType(), nullable=True),
    T.StructField(name="source_id", dataType=T.IntegerType(), nullable=True),
    T.StructField(name="row_update_dtime", dataType=T.TimestampType(), nullable=True),
    T.StructField(name="row_loading_id", dataType=T.LongType(), nullable=True),
    T.StructField(name="row_hash_val", dataType=T.StringType(), nullable=True),
])


In [ ]:
# Читаем 2 monthly-среза и 2 daily-среза.
ccd_2023 = spark.read.parquet(str(BASE_DIR / "client_cards_daily/row_actual_to='2023-04-03'"))
ccd_9999 = spark.read.parquet(str(BASE_DIR / "client_cards_daily/row_actual_to='9999-12-31'"))
cci_02 = spark.read.parquet(str(BASE_DIR / "credit_cards_info/report_dt='2023-02-28'"))
cci_03 = spark.read.parquet(str(BASE_DIR / "credit_cards_info/report_dt='2023-03-31'"))
dci_02 = spark.read.parquet(str(BASE_DIR / "deb_cards_info/report_dt='2023-02-28'"))
dci_03 = spark.read.parquet(str(BASE_DIR / "deb_cards_info/report_dt='2023-03-31'"))

# Собираем единый список сырых client id из всех источников.
client_keys = (
    ccd_2023.select(F.col("id").cast("string").alias("raw_client_id"))
    .unionByName(other=ccd_9999.select(F.col("id").cast("string").alias("raw_client_id")))
    .unionByName(other=cci_02.select(F.col("id").cast("string").alias("raw_client_id")))
    .unionByName(other=cci_03.select(F.col("id").cast("string").alias("raw_client_id")))
    .unionByName(other=dci_02.select(F.col("id").cast("string").alias("raw_client_id")))
    .unionByName(other=dci_03.select(F.col("id").cast("string").alias("raw_client_id")))
    .distinct()
)
# Назначаем каждому сырому id суррогатный целочисленный ключ.
CLIENT_KEY_DF = client_keys.withColumn(
    colName="client_id",
    col=F.row_number().over(sql.Window.orderBy("raw_client_id")).cast("int"),
)

def stack_expr(names: list[str]) -> str:
    """Строит Spark stack-выражение для разворота wide-таблицы в EAV."""
    values = ", ".join(f"'{name}', `{name}`" for name in names)
    return f"stack({len(names)}, {values}) as (attribute_name, attribute_value)"

def infer_data_type(name: str) -> str:
    """Грубо определяет бизнес-тип атрибута по имени колонки."""
    if name.endswith("_dt") or name.endswith("_from") or name.endswith("_to"):
        return "DATE"
    if name.endswith("_amt"):
        return "DECIMAL"
    if name.endswith("_nflag") or name.endswith("_nfalg") or name.endswith("_qty"):
        return "INT"
    return "STRING"

def build_dim_sources() -> sql.DataFrame:
    """Собирает справочник источников dim_sources."""
    rows = [
        (
            meta["source_id"],
            source_name,
            meta["source_description"],
            meta["update_frequency"],
            NOW_TS,
            NOW_TS,
            START_TS,
            FUTURE_TS,
        )
        for source_name, meta in SOURCE_REGISTRY.items()
    ]
    return spark.createDataFrame(data=rows, schema=dim_sources_schema)

def build_dim_attributes() -> sql.DataFrame:
    """Собирает справочник атрибутов dim_attributes."""
    rows = []
    for source_name, meta in SOURCE_REGISTRY.items():
        for column_name in meta["columns"]:
            rows.append((
                ATTR_ID_BY_NAME[column_name],
                column_name,
                f"Attribute {column_name}",
                infer_data_type(column_name),
                meta["source_id"],
                meta["update_frequency"],
                NOW_TS,
                NOW_TS,
            ))
    return spark.createDataFrame(data=rows, schema=dim_attributes_schema)

def unpivot_table(df: sql.DataFrame, source_name: str) -> sql.DataFrame:
    """Разворачивает источник в EAV-таблицу и приводит поля к целевой схеме."""
    meta = SOURCE_REGISTRY[source_name]
    stack = F.expr(stack_expr(list(meta["columns"])))
    prepared_source = (
        # Привязываем surrogate client_id и приводим техполя к нужным типам.
        df.withColumn(colName="raw_client_id", col=F.col("id").cast("string"))
        .join(other=CLIENT_KEY_DF, on="raw_client_id", how="left")
        .select(
            F.col("client_id"),
            *[F.col(column_name).cast("string").alias(column_name) for column_name in meta["columns"]],
            F.col("row_update_dtime").cast("timestamp").alias("row_update_dtime"),
            F.col("loading_id").cast("bigint").alias("row_loading_id"),
            F.col("row_hash_val").cast("string").alias("row_hash_val"),
            *(
                [F.col("report_dt").cast("string").alias("report_dt")]
                if meta["update_frequency"] == "monthly"
                else [
                    F.col("row_actual_from").cast("string").alias("row_actual_from"),
                    F.col("row_actual_to").cast("string").alias("row_actual_to"),
                ]
            ),
        )
    )
    prepared = prepared_source.select(
        # Разворачиваем бизнес-колонки в пары attribute_name / attribute_value.
        F.col("client_id"),
        stack,
        F.col("row_update_dtime"),
        F.col("row_loading_id"),
        F.col("row_hash_val"),
        *(
            [F.col("report_dt")]
            if meta["update_frequency"] == "monthly"
            else [F.col("row_actual_from"), F.col("row_actual_to")]
        ),
    ).withColumn(
        colName="attribute_id",
        col=reduce(
            lambda acc, item: F.when(F.col("attribute_name") == F.lit(item[0]), F.lit(item[1])).otherwise(acc),
            ATTR_ID_BY_NAME.items(),
            F.lit(None),
        ).cast("int"),
    ).withColumn(colName="attribute_value", col=F.col("attribute_value").cast("string")).withColumn(colName="source_id", col=F.lit(meta["source_id"]))
    # Возвращаем monthly- или daily-проекцию в целевую таблицу.
    if meta["update_frequency"] == "monthly":
        return prepared.select(
            "client_id",
            "attribute_id",
            "report_dt",
            "attribute_value",
            "source_id",
            "row_update_dtime",
            "row_loading_id",
            "row_hash_val",
        )
    return prepared.select(
        "client_id",
        "attribute_id",
        "attribute_value",
        "row_actual_from",
        "row_actual_to",
        "source_id",
        "row_update_dtime",
        "row_loading_id",
        "row_hash_val",
    )


In [ ]:
# Собираем итоговые таблицы хранилища.
dim_sources = build_dim_sources()
dim_attributes = build_dim_attributes()

# Monthly-источники складываем в SCD1-таблицу.
client_monthly_attrs_scd1 = (
    unpivot_table(cci_02, "credit_cards_info")
    .unionByName(other=unpivot_table(cci_03, "credit_cards_info"))
    .unionByName(other=unpivot_table(dci_02, "deb_cards_info"))
    .unionByName(other=unpivot_table(dci_03, "deb_cards_info"))
    .dropDuplicates(subset=["client_id", "attribute_id", "report_dt"])
)

# Daily-источник складываем в SCD2-таблицу.
client_daily_attrs_scd2 = (
    unpivot_table(ccd_2023, "client_cards_daily")
    .unionByName(other=unpivot_table(ccd_9999, "client_cards_daily"))
    .dropDuplicates(subset=["client_id", "attribute_id", "row_actual_from"])
)

# Фиксируем 6 фактов загрузки в журнале.
load_log_rows = [
    (1, 1, "2023-04-03", NOW_TS, NOW_TS, "client_daily_attrs_scd2", "success", ccd_2023.count(), None),
    (2, 1, "9999-12-31", NOW_TS, NOW_TS, "client_daily_attrs_scd2", "success", ccd_9999.count(), None),
    (3, 2, "2023-02-28", NOW_TS, NOW_TS, "client_monthly_attrs_scd1", "success", cci_02.count(), None),
    (4, 2, "2023-03-31", NOW_TS, NOW_TS, "client_monthly_attrs_scd1", "success", cci_03.count(), None),
    (5, 3, "2023-02-28", NOW_TS, NOW_TS, "client_monthly_attrs_scd1", "success", dci_02.count(), None),
    (6, 3, "2023-03-31", NOW_TS, NOW_TS, "client_monthly_attrs_scd1", "success", dci_03.count(), None),
]
load_log = spark.createDataFrame(data=load_log_rows, schema=load_log_schema)

# Записываем parquet-таблицы в warehouse.
WAREHOUSE_ROOT.mkdir(parents=True, exist_ok=True)
dim_sources.write.mode("overwrite").parquet(str(WAREHOUSE_ROOT / "dim_sources"))
dim_attributes.write.mode("overwrite").parquet(str(WAREHOUSE_ROOT / "dim_attributes"))
load_log.write.mode("overwrite").parquet(str(WAREHOUSE_ROOT / "load_log"))
client_monthly_attrs_scd1.write.mode("overwrite").parquet(str(WAREHOUSE_ROOT / "client_monthly_attrs_scd1"))
client_daily_attrs_scd2.write.mode("overwrite").parquet(str(WAREHOUSE_ROOT / "client_daily_attrs_scd2"))

# Показываем размеры итоговых таблиц.
{
    "dim_sources": dim_sources.count(),
    "dim_attributes": dim_attributes.count(),
    "load_log": load_log.count(),
    "client_monthly_attrs_scd1": client_monthly_attrs_scd1.count(),
    "client_daily_attrs_scd2": client_daily_attrs_scd2.count(),
}


In [ ]:
# Показываем несколько строк из каждой итоговой таблицы.
print("dim_sources")
dim_sources.orderBy("source_id").show(10, truncate=False)

print("dim_attributes")
dim_attributes.orderBy("attribute_id").show(10, truncate=False)

print("load_log")
load_log.orderBy("load_id").show(10, truncate=False)

print("client_monthly_attrs_scd1")
client_monthly_attrs_scd1.orderBy("client_id", "attribute_id", "report_dt").show(10, truncate=False)

print("client_daily_attrs_scd2")
client_daily_attrs_scd2.orderBy("client_id", "attribute_id", "row_actual_from").show(10, truncate=False)


In [ ]:
# Debug: проверяем, какие пути и партиции реально видны в окружении.
print("BASE_DIR =", BASE_DIR)
print("client_cards_daily exists:", (BASE_DIR / "client_cards_daily").exists())
print("credit_cards_info exists:", (BASE_DIR / "credit_cards_info").exists())
print("deb_cards_info exists:", (BASE_DIR / "deb_cards_info").exists())

print("\nclient_cards_daily partitions:")
for path in sorted((BASE_DIR / "client_cards_daily").glob("*"))[:20]:
    print(path)

print("\ncredit_cards_info partitions:")
for path in sorted((BASE_DIR / "credit_cards_info").glob("*"))[:20]:
    print(path)

print("\ndeb_cards_info partitions:")
for path in sorted((BASE_DIR / "deb_cards_info").glob("*"))[:20]:
    print(path)

print("\nclient_cards_daily/row_actual_to='2023-04-03' contents:")
for path in sorted((BASE_DIR / "client_cards_daily/row_actual_to='2023-04-03'").glob("*"))[:20]:
    print(path)

print("\nclient_cards_daily/row_actual_to='9999-12-31' contents:")
for path in sorted((BASE_DIR / "client_cards_daily/row_actual_to='9999-12-31'").glob("*"))[:20]:
    print(path)

print("\ncredit_cards_info/report_dt='2023-02-28' contents:")
for path in sorted((BASE_DIR / "credit_cards_info/report_dt='2023-02-28'").glob("*"))[:20]:
    print(path)

print("\ncredit_cards_info/report_dt='2023-03-31' contents:")
for path in sorted((BASE_DIR / "credit_cards_info/report_dt='2023-03-31'").glob("*"))[:20]:
    print(path)

print("\ndeb_cards_info/report_dt='2023-02-28' contents:")
for path in sorted((BASE_DIR / "deb_cards_info/report_dt='2023-02-28'").glob("*"))[:20]:
    print(path)

print("\ndeb_cards_info/report_dt='2023-03-31' contents:")
for path in sorted((BASE_DIR / "deb_cards_info/report_dt='2023-03-31'").glob("*"))[:20]:
    print(path)
